In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/twcs.csv")

In [2]:
df["in_response_to_tweet_id"] = pd.to_numeric(
    df["in_response_to_tweet_id"],
    errors="coerce"
).astype("Int64")

In [3]:
parent = dict(
    zip(
        df["tweet_id"],
        df["in_response_to_tweet_id"]
    )
)


def find_root(tweet_id):
    visited = set()

    while pd.notna(parent.get(tweet_id)):

        if tweet_id in visited:
            break

        visited.add(tweet_id)

        tweet_id = int(parent[tweet_id])

    return tweet_id


df["conversation_id"] = df["tweet_id"].map(find_root)

In [4]:
spotify_conversation_ids = df[
    df["author_id"] == "SpotifyCares"
]["conversation_id"].dropna().unique()


spotify_df = df[
    df["conversation_id"].isin(spotify_conversation_ids)
].copy()

In [5]:
print("Shape:")
print(spotify_df.shape)

print("\nNumber of unique conversations:")
print(spotify_df["conversation_id"].nunique())

print("\nInbound value counts:")
print(spotify_df["inbound"].value_counts())

print("\nTop 10 authors:")
print(spotify_df["author_id"].value_counts().head(10))

Shape:
(91889, 8)

Number of unique conversations:
28280

Inbound value counts:
inbound
True     48543
False    43346
Name: count, dtype: int64

Top 10 authors:
author_id
SpotifyCares    43265
115888            332
hulu_support       49
125633             28
287348             24
215073             21
176622             21
AppleSupport       19
158590             19
220017             18
Name: count, dtype: int64


In [6]:
other_outbound_authors = (
    spotify_df[
        (spotify_df["inbound"] == False) &
        (spotify_df["author_id"] != "SpotifyCares")
    ]["author_id"]
    .value_counts()
)

print("Number of other outbound authors:",
      other_outbound_authors.shape[0])

print(other_outbound_authors.head(30))

Number of other outbound authors: 7
author_id
hulu_support       49
AppleSupport       19
AmazonHelp          7
comcastcares        2
British_Airways     2
asksalesforce       1
O2                  1
Name: count, dtype: int64


In [8]:
spotify_df["created_at"] = pd.to_datetime(
    spotify_df["created_at"],
    utc=True
)
print(spotify_df["created_at"].dtype)
print(spotify_df["created_at"].head())

C:\Users\Nihelesh M U\AppData\Local\Temp\ipykernel_18344\1583779328.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  spotify_df["created_at"] = pd.to_datetime(


datetime64[ns, UTC]
540   2017-10-31 22:28:16+00:00
541   2017-10-31 23:36:20+00:00
542   2017-10-31 23:39:03+00:00
543   2017-10-31 21:41:37+00:00
544   2017-10-31 21:04:13+00:00
Name: created_at, dtype: datetime64[ns, UTC]


In [13]:
spotify_df = spotify_df.sort_values(
    ["conversation_id", "created_at"]
).copy()

spotify_df["turn_number"] = (
    spotify_df.groupby("conversation_id").cumcount() + 1
)


In [14]:
spotify_df

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,conversation_id,turn_number
547,855,115887,True,2017-10-31 19:10:51+00:00,i’m pissed my @115888 shuffle and repeat butto...,854,<NA>,855,1
546,854,SpotifyCares,False,2017-10-31 19:36:16+00:00,"@115887 Hey! What device, operating system, an...",853,855,855,2
545,853,115887,True,2017-10-31 19:57:59+00:00,@SpotifyCares iphone 7+ and i have the most re...,852,854,855,3
544,852,SpotifyCares,False,2017-10-31 21:04:13+00:00,"@115887 Thanks. Just to be sure, are you Free ...",850,853,855,4
543,850,115887,True,2017-10-31 21:41:37+00:00,@SpotifyCares Premium &amp; when i️ have it on...,848,852,855,5
...,...,...,...,...,...,...,...,...,...
2811214,2987398,SpotifyCares,False,2017-11-30 07:34:24+00:00,@823514 Can you send us a DM with your account...,NaN,2987399,2987399,2
2811217,2987401,823709,True,2017-11-30 07:20:48+00:00,"@SpotifyCares Hey Guys, my family members cann...",2987400,<NA>,2987401,1
2811216,2987400,SpotifyCares,False,2017-11-30 07:29:24+00:00,@823709 Hey Benedict! Can you DM us yours and ...,NaN,2987401,2987401,2
2811504,2987685,823786,True,2017-10-31 22:01:40+00:00,@SpotifyCares I tried on many devices and brow...,2987684,<NA>,2987685,1


In [15]:
url_count = spotify_df["text"].str.contains(
    r"http|https|t\.co",
    case=False,
    regex=True,
    na=False
).sum()

mention_count = spotify_df["text"].str.contains(
    r"@\w+",
    regex=True,
    na=False
).sum()

newline_count = spotify_df["text"].str.contains(
    r"\n",
    regex=True,
    na=False
).sum()

print("Tweets containing URLs:", url_count)
print("Tweets containing mentions:", mention_count)
print("Tweets containing newlines:", newline_count)

print("Missing text:", spotify_df["text"].isna().sum())

print(
    "Empty text:",
    (spotify_df["text"].str.strip() == "").sum()
)

Tweets containing URLs: 28280
Tweets containing mentions: 90311
Tweets containing newlines: 1784
Missing text: 0
Empty text: 0


In [16]:
mention_examples = spotify_df[
    spotify_df["text"].str.contains(
        r"@\w+",
        regex=True,
        na=False
    )
]["text"].head(20)

for text in mention_examples:
    print(text)
    print("-" * 100)

i’m pissed my @115888 shuffle and repeat button just don’t fucking work and i’m getting frustrated
----------------------------------------------------------------------------------------------------
@115887 Hey! What device, operating system, and Spotify version are you using? We'll see what we can suggest /CB
----------------------------------------------------------------------------------------------------
@SpotifyCares iphone 7+ and i have the most recent update for spotify
----------------------------------------------------------------------------------------------------
@115887 Thanks. Just to be sure, are you Free or Premium? Also, can you give us more info on what happens when you try using it? /CB
----------------------------------------------------------------------------------------------------
@SpotifyCares Premium &amp; when i️ have it on shuffle it turns off when the song is done and just plays in order and the repeat lights up but doesn’t repeat
-----------------------

In [17]:
company_mentions = (
    spotify_df["text"]
    .str.findall(r"@\w+")
    .explode()
    .value_counts()
)

print(company_mentions.head(30))

text
@SpotifyCares    31196
@115888          15375
@117153            963
@116130            547
@117168            449
@125633            349
@spotifycares      346
@118266            300
@147836            189
@115948            173
@118062            127
@116380             96
@3493               88
@115940             79
@148611             69
@151631             61
@AppleSupport       55
@hulu_support       52
@127972             49
@115858             46
@515265             44
@127637             43
@191335             43
@129941             41
@131098             40
@7997               35
@181186             34
@121907             34
@137949             33
@133324             32
Name: count, dtype: int64


In [18]:
import re
import html

In [19]:
def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # Decode HTML entities
    text = html.unescape(text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # Remove Twitter agent signatures
    text = re.sub(r"\^[A-Za-z]{1,4}\b", "", text)

    # Remove emojis
    emoji_pattern = re.compile(
        "["
        "\U0001F1E0-\U0001F1FF"
        "\U0001F300-\U0001F5FF"
        "\U0001F600-\U0001F64F"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FAFF"
        "\U00002700-\U000027BF"
        "\U00002600-\U000026FF"
        "]+",
        flags=re.UNICODE
    )

    text = emoji_pattern.sub("", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [20]:
test_texts = [
    "@115770 I have an order problem https://t.co/example",
    "@SpotifyCares My order hasn't arrived 😭",
    "@UPSHelp Please check my package",
    "Hello\n\nI need help with my refund",
    "   My order     is delayed   "
]

for text in test_texts:
    print("BEFORE:", text)
    print("AFTER :", clean_text(text))
    print("-" * 80)

BEFORE: @115770 I have an order problem https://t.co/example
AFTER : @115770 I have an order problem
--------------------------------------------------------------------------------
BEFORE: @SpotifyCares My order hasn't arrived 😭
AFTER : @SpotifyCares My order hasn't arrived
--------------------------------------------------------------------------------
BEFORE: @UPSHelp Please check my package
AFTER : @UPSHelp Please check my package
--------------------------------------------------------------------------------
BEFORE: Hello

I need help with my refund
AFTER : Hello I need help with my refund
--------------------------------------------------------------------------------
BEFORE:    My order     is delayed   
AFTER : My order is delayed
--------------------------------------------------------------------------------


In [21]:
spotify_df["text"] = spotify_df["text"].apply(clean_text)

In [22]:
! pip install langdetect


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

def detect_language(text):
    try:
        return detect(text)
    except:
        return "unknown"

sample = spotify_df["text"].dropna().sample(
    n=min(10000, len(spotify_df)),
    random_state=42
)

sample_languages = sample.apply(detect_language)

sample_languages.value_counts()

text
en         9754
fr           46
id           26
tl           15
es           15
nl           11
af           11
sv           10
de           10
ca            9
pt            9
cy            9
unknown       9
it            8
pl            7
tr            7
da            5
no            5
th            4
sw            3
et            3
sk            3
hr            3
so            3
vi            3
sl            3
fi            2
lt            2
cs            2
ro            2
ar            1
Name: count, dtype: int64

In [27]:
remaining_urls = spotify_df["text"].str.contains(
    r"http|https|t\.co",
    case=False,
    regex=True,
    na=False
).sum()

print("URLs remaining:", remaining_urls)

remaining_user_mentions = spotify_df["text"].str.contains(
    r"@\d+",
    regex=True,
    na=False
).sum()

print("Numeric mentions remaining:", remaining_user_mentions)

remaining_newlines = spotify_df["text"].str.contains(
    r"\n",
    regex=True,
    na=False
).sum()

print("Newlines remaining:", remaining_newlines)

empty_clean_text = (
    spotify_df["text"]
    .str.strip()
    .eq("")
    .sum()
)

print("Empty cleaned tweets:", empty_clean_text)

URLs remaining: 14
Numeric mentions remaining: 63014
Newlines remaining: 0
Empty cleaned tweets: 22


In [28]:
remaining_urls = spotify_df["text"].str.contains(
    r"https?://\S+",
    regex=True,
    na=False
).sum()

print("Actual URLs remaining:", remaining_urls)

Actual URLs remaining: 0


In [31]:
amazon_clean = spotify_df[
    spotify_df["text"].str.strip() != ""
].copy()

In [32]:
print("Conversations before:", spotify_df["conversation_id"].nunique())
print("Conversations after:", amazon_clean["conversation_id"].nunique())

Conversations before: 28280
Conversations after: 28280


In [34]:
from langdetect import DetectorFactory

# Make language detection reproducible
DetectorFactory.seed = 42

print("\nDetecting languages...")

spotify_df["language"] = spotify_df["text"].apply(
    detect_language
)

print("\nLanguage distribution:")
print(spotify_df["language"].value_counts())

# Keep only English
spotify_df = spotify_df[
    spotify_df["language"] == "en"
].copy()

print("\nEnglish tweets:", len(spotify_df))
print(
    "English conversations:",
    spotify_df["conversation_id"].nunique()
)

print("\nVerification:")
print(
    spotify_df[
        ["tweet_id", "text", "language"]
    ].head(20)
)


Detecting languages...

Language distribution:
language
en         89477
fr           411
id           348
tl           132
af           128
de           118
nl           112
sv           112
es            98
ca            90
unknown       90
it            87
no            67
tr            64
so            64
pt            62
cy            57
da            57
th            42
sk            37
fi            33
pl            32
ro            27
et            24
vi            22
sw            21
sq            15
cs            14
sl            11
hr             9
ja             7
lt             4
ar             4
hu             4
lv             3
ko             3
ru             1
fa             1
bg             1
Name: count, dtype: int64

English tweets: 89477
English conversations: 28241

Verification:
         tweet_id                                               text language
547           855  i’m pissed my @115888 shuffle and repeat butto...       en
546           854  @115887 Hey!

In [36]:
print(spotify_df["conversation_id"].nunique())

28241


In [37]:
# Remove language column
spotify_df = spotify_df.drop(
    columns=["language"],
    errors="ignore"
)

# Save CSV
spotify_df.to_csv(
    "../data/processed/SpotifyCares.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ SpotifyCares dataset saved without language column!")

✅ SpotifyCares dataset saved without language column!
